# Phase 6 — which network, measured rather than assumed

Four training runs that differ in the network and in nothing else. The baseline
is a U-Net with a ResNet-34 encoder because that is where the project started,
not because anything measured said so, and this notebook is what replaces the
assumption with a number.

| run | decoder | encoder | what it isolates |
|---|---|---|---|
| a | Unet | resnet34 | the reference, at this epoch count |
| b | Unet | tu-convnext_tiny | the encoder alone |
| c | UPerNet | tu-convnext_tiny | the decoder, on b's encoder |
| d | Segformer | mit_b2 | a different family |

Why these and not others. Filaments are thin, they cover 0.35% of a frame, and
the ones missing from the probability map entirely are the small ones: a
correct box drawn around 266 of fold 0's filaments contains no pixel above 0.05.
An encoder-decoder that halves its resolution five times has the most to lose
there. UPerNet pools context over the whole frame before decoding; the
Segformer encoder attends globally from its first stage. Both are reasons about
this data, not about publication dates.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Add the competition data as an input
4. **Save Version** with *Save output* on, or the checkpoints and maps are discarded

Expected wall clock on one T4: roughly 50 minutes for run a and 60 to 75 for
each of the others, so about four hours in total, plus twenty minutes writing
probability maps.

## 1. Clone the repository and put it on the path

The repository is cloned rather than installed, because `configs/paths.yaml`
and the frozen splits in `configs/splits/` sit beside the package rather than
inside it. Set `REF` to the branch or commit this run is to be reproducible
from.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase6-architecture"  # branch or commit hash
CHECKOUT = "/kaggle/working/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())
if torch.cuda.is_available():
    print("name:", torch.cuda.get_device_name(0))

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

dataset_root = candidates[0].parent.parent
os.environ["MAGFILO_ROOT"] = str(dataset_root)
print("MAGFILO_ROOT =", dataset_root)

paths = load_paths().require_dataset()
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))

## 3. The four runs

The ground truth is encoded once and handed to every scoring call: it takes as
long as the inference and does not depend on the model.

Scoring uses the post-processing Phase 3 settled on — threshold 0.5, minimum
area 400, rejoining within 24 pixels — for all four runs. The defaults in the
code are not that configuration, so they are named explicitly here. Two runs
are comparable only when the chain behind them is the same.

In [ ]:
from dataclasses import replace

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.training.config import TrainConfig

RUNS = ["a_unet_resnet34", "b_unet_convnext", "c_upernet_convnext", "d_segformer_mit"]

# The configuration Phase 3 settled on, applied identically to every run.
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0}

dataset = load_annotations(paths.train_annotations)
val_stems = load_fold(0, f"{CHECKOUT}/configs/splits").val
print(f"fold 0: {len(val_stems)} validation frames")

configs = {}
for name in RUNS:
    configs[name] = replace(
        TrainConfig.from_yaml(f"{CHECKOUT}/configs/phase6/{name}.yaml"),
        num_workers=2,
        output_dir=Path(f"/kaggle/working/{name}"),
    )
    config = configs[name]
    print(f"{name:<22} {config.architecture:<10} {config.encoder:<18} {config.epochs} epochs")

In [ ]:
from filament.submit.rle import masks_to_gt_df, read_submission, write_submission

gt_path = Path("/kaggle/working/gt_fold0.csv")
if gt_path.exists():
    gt_df = read_submission(gt_path)
else:
    gt_df = masks_to_gt_df(dataset, val_stems)
    write_submission(gt_df, gt_path)
print(f"{len(gt_df)} ground-truth filaments")

In [ ]:
import json
import logging
import time

from filament.evaluation import evaluate
from filament.training.loop import load_checkpoint, train

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

results = {}
for name in RUNS:
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    started = time.perf_counter()
    outcome = train(configs[name])
    training_minutes = (time.perf_counter() - started) / 60

    model, _ = load_checkpoint(outcome.checkpoint)
    evaluation, _ = evaluate(
        model,
        dataset,
        paths.train_images,
        val_stems,
        size=configs[name].image_size,
        device="cuda",
        gt_df=gt_df,
        **SCORING,
    )
    results[name] = evaluation.to_dict() | {
        "architecture": configs[name].architecture,
        "encoder": configs[name].encoder,
        "best_epoch": outcome.best_epoch,
        "best_val_loss": round(outcome.best_val_loss, 4),
        "training_minutes": round(training_minutes, 1),
    }
    Path(f"/kaggle/working/{name}/eval_fold0.json").write_text(json.dumps(results[name], indent=2))
    print(evaluation)
    print(f"{name}: {training_minutes:.1f} min")

    # Freeing it matters: four networks held at once will not fit alongside the
    # next run's activations.
    del model
    torch.cuda.empty_cache()

Path("/kaggle/working/phase6_summary.json").write_text(json.dumps(results, indent=2))

## 4. The comparison

The reference for all of this is **PQ 0.3756**, the Phase 3 configuration at
forty epochs. Run *a* is the same network at twenty, so the gap between them is
the epoch count and says how much of any difference below is simply schedule.

Adoption needs **+0.01 over run a** and an explanation of why it moved. SQ and
RQ are shown apart because they say different things: SQ is how well a matched
filament is drawn, RQ is how many were matched at all. Nothing tried so far has
moved SQ out of 0.645 to 0.665, and whether any of these does is the question
this notebook exists to answer.

In [ ]:
import pandas as pd

table = pd.DataFrame(results).T[
    [
        "architecture",
        "encoder",
        "pq",
        "sq",
        "rq",
        "tp",
        "fp",
        "fn",
        "fused",
        "split",
        "best_epoch",
        "training_minutes",
    ]
]
table["pq_vs_a"] = (table["pq"] - table.loc["a_unet_resnet34", "pq"]).round(4)
table

## 5. Probability maps for the best run

Saved only for the winner: the post-processing sweeps run on the maps rather
than on the model, and doing that offline means no further GPU time is needed
to tune the chain behind whichever network wins.

Float16 keeps a fold to about 300 MB.

In [ ]:
import numpy as np

from filament.data.image import load_grayscale
from filament.evaluation import predict_probability

best = max(results, key=lambda name: results[name]["pq"])
print(f"best: {best}  PQ {results[best]['pq']}")

model, _ = load_checkpoint(Path(f"/kaggle/working/{best}/best.pt"))
maps_dir = Path(f"/kaggle/working/prob_fold0_{best}")
maps_dir.mkdir(parents=True, exist_ok=True)

for position, stem in enumerate(val_stems, start=1):
    probability = predict_probability(
        model,
        load_grayscale(paths.train_images / f"{stem}.jpeg"),
        size=configs[best].image_size,
        device="cuda",
    )
    np.save(maps_dir / f"{stem}.npy", probability.astype(np.float16))
    if position % 40 == 0:
        print(f"{position}/{len(val_stems)}")

written = sorted(maps_dir.glob("*.npy"))
print(f"{len(written)} maps, {sum(p.stat().st_size for p in written) / 1e6:.0f} MB")

## 6. What to record

Into the lab notebook, for every one of the four runs and not only the winner —
the ones that did not work are what the ablation table in the final report is
made of:

- PQ, SQ, RQ, TP, FP, FN, and the fused and split counts
- best epoch and training time
- the commit hash this notebook cloned

Do not submit from here. The comparison is decided on fold 0 locally; folds 1
to 4 stay untouched until a configuration is frozen.